# Idealista

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# Ruta al archivo Excel en Google Drive

spreadsheet_url = '/content/drive/MyDrive/TFM/Fuentes/ProcesadosBCN_Prep_Data_23_08_2024.xlsx'
# Leer el archivo Excel
df_viviendas = pd.read_excel(spreadsheet_url)

df_viviendas = df_viviendas.loc[:, ~df_viviendas.columns.str.contains('Unnamed')]

# Mostrar las primeras filas del DataFrame
df_viviendas.head()


,index,url,floor,price,propertyType,operation,size,exterior,rooms,bathrooms,...,country,neighborhood,latitude,longitude,status,newDevelopment,hasLift,topNewDevelopment,isParkingSpaceIncludedInPrice,ParkingSpacePrice
0,1,https://www.idealista.com/inmueble/99908514/,2.0,2112,flat,rent,70,False,2,1,...,es,Sant Antoni,"413,766,344","21,605,432",good,False,True,False,False,NaN
1,2,https://www.idealista.com/inmueble/103626561/,NaN,2200,chalet,rent,81,False,2,3,...,es,Vallcarca i els Penitents,"414,201,952","21,415,648",good,False,False,False,True,NaN
2,3,https://www.idealista.com/inmueble/104316103/,7.0,2050,flat,rent,120,True,3,3,...,es,La Dreta de l'Eixample,"413,939,681","21,670,973",good,False,True,False,False,NaN
3,4,https://www.idealista.com/inmueble/105181609/,4.0,1500,flat,rent,65,True,2,1,...,es,Sants - Badal,"413,700,473","21,309,462",good,False,True,False,False,NaN
4,5,https://www.idealista.com/inmueble/104961590/,12.0,6600,flat,rent,274,True,4,3,...,es,Sant Gervasi - Galvany,"413,939,701","21,385,965",good,False,True,False,True,NaN


Comprobamos si hay Nan en las columnas de longitude y latitude:

In [ ]:
df_viviendas = df_viviendas.dropna(subset=['longitude', 'latitude'])

# Filtrar filas donde 'longitude' o 'latitude' son NaN
missing_values = df_viviendas[df_viviendas[['longitude', 'latitude']].isna().any(axis=1)]

# Mostrar las filas con valores NaN en 'longitude' o 'latitude'
print(missing_values)

Empty DataFrame
Columns: [index, url, floor, price, propertyType, operation, size, exterior, rooms, bathrooms, address, province, municipality, district, country, neighborhood, latitude, longitude, status, newDevelopment, hasLift, topNewDevelopment, isParkingSpaceIncludedInPrice, ParkingSpacePrice]
Index: []

[0 rows x 24 columns]


In [ ]:
import pandas as pd
from geopy.distance import geodesic

# Amenities

# Cálculo de distancias (Euclidiana)

## Bancos

In [ ]:
# Cargar el archivo CSV
spreadsheet_url = '/content/drive/MyDrive/TFM/Fuentes/Data/bcn_bancos.csv'
df_bancos = pd.read_csv(spreadsheet_url)

df_bancos.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/TFM/Fuentes/Data/bcn_bancos.csv'

In [ ]:
# Añadir una columna para la distancia mínima y el banco más cercano
df_viviendas['distancia_minima_banco'] = float('inf')
df_viviendas['banco_mas_cercano'] = None

#Número de bancos cercanos:
df_viviendas['numero_bancos_cercanos'] = 0

# Calcular la distancia geodésica de cada vivienda a cada banco
for i, vivienda in df_viviendas.iterrows():
    lat_viv, lon_viv = vivienda['latitude'], vivienda['longitude']
    distancias = df_bancos.apply(lambda banco: geodesic((lat_viv, lon_viv), (banco['latitude'], banco['longitude'])).meters, axis=1)


#Encontrar el banco mas cercano:
    idx_min = distancias.idxmin()
    df_viviendas.at[i, 'distancia_minima_banco'] = distancias.min()
    df_viviendas.at[i, 'banco_mas_cercano'] = df_bancos.at[idx_min, 'amenity']  # Ajusta según el nombre de la columna correspondiente

# Contar el número de bancos a menos de 500 metros
    num_bancos_cercanos = (distancias <= 500).sum()
    df_viviendas.at[i, 'numero_bancos_cercanos'] = num_bancos_cercanos


# Guardar el archivo actualizado
df_viviendas.to_excel('/content/drive/MyDrive/TFM/Fuentes/ProcesadosBCN_Prep_Data_23_08_2024.xlsx', index=False)

print("Proceso completado y archivo guardado correctamente.")

Proceso completado y archivo guardado correctamente.


In [ ]:

df_viviendas.head(10)

,index,Fecha extraccion,floor,price,propertyType,operation,size,exterior,rooms,bathrooms,...,longitude,status,newDevelopment,hasLift,topNewDevelopment,isParkingSpaceIncludedInPrice,ParkingSpacePrice,distancia_minima_banco,banco_mas_cercano,numero_bancos_cercanos
0,1,2382024,3,1500,flat,rent,80.0,1.0,3,2,...,2.391546,good,False,True,False,NaN,NaN,15158.966797,bank,0
1,2,2382024,1,1496,flat,rent,55.0,NaN,2,2,...,2.120859,good,False,True,False,1.0,NaN,336.168215,bank,8
2,3,2382024,2,1240,flat,rent,66.0,1.0,1,1,...,2.124236,good,False,False,False,NaN,NaN,224.807509,bank,5
3,4,2382024,1,750,studio,rent,30.0,0.0,0,1,...,2.122919,good,False,False,False,NaN,NaN,353.780818,bank,5
4,5,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,2.180652,good,False,False,False,NaN,NaN,194.327806,bank,7
5,6,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,2.180652,good,False,False,False,NaN,NaN,194.327806,bank,7
6,7,2382024,NaN,1050,studio,rent,15.0,NaN,0,1,...,2.177981,good,False,False,False,NaN,NaN,86.257947,bank,19
7,8,2382024,1,1275,flat,rent,39.0,NaN,1,1,...,2.174292,good,False,False,False,NaN,NaN,186.870914,bank,8
8,9,2382024,3,622,flat,rent,44.0,1.0,1,1,...,2.215093,good,False,False,False,NaN,NaN,183.195953,bank,4
9,10,2382024,2,1500,studio,rent,35.0,1.0,0,1,...,2.189523,good,False,False,False,NaN,NaN,232.317274,bank,4


## Starbucks

In [ ]:
# Cargar el archivo CSV
spreadsheet_url = '/content/drive/MyDrive/TFM/Fuentes/Data/bcn_starbucks.csv'
df_starbucks = pd.read_csv(spreadsheet_url)

df_starbucks.head()

,name,longitude,latitude
0,Starbucks,2.177192,41.385291
1,Starbucks,2.159688,41.395939
2,Starbucks,2.215722,41.409540
3,Starbucks,2.136377,41.391192
4,Starbucks,2.163245,41.385259


In [ ]:
# Añadir una columna para la distancia mínima y el banco más cercano
df_viviendas['distancia_minima_starbucks'] = float('inf')
df_viviendas['starbucks_mas_cercano'] = None

#Número de bancos cercanos:
df_viviendas['numero_starbucks_cercanos'] = 0

# Calcular la distancia geodésica de cada vivienda a cada banco
for i, vivienda in df_viviendas.iterrows():
    lat_viv, lon_viv = vivienda['latitude'], vivienda['longitude']
    distancias = df_starbucks.apply(lambda starbucks: geodesic((lat_viv, lon_viv), (starbucks['latitude'], starbucks['longitude'])).meters, axis=1)


#Encontrar el banco mas cercano:
    idx_min = distancias.idxmin()
    df_viviendas.at[i, 'distancia_minima_starbucks'] = distancias.min()
    df_viviendas.at[i, 'starbucks_mas_cercano'] = df_starbucks.at[idx_min, 'name']  # Ajusta según el nombre de la columna correspondiente

# Contar el número de bancos a menos de 500 metros
    num_starbucks_cercanos = (distancias <= 500).sum()
    df_viviendas.at[i, 'numero_starbucks_cercanos'] = num_starbucks_cercanos


# Guardar el archivo actualizado
df_viviendas.to_excel('/content/drive/MyDrive/TFM/Fuentes/Procesados/BCN_Prep_Data_23_08_2024.xlsx', index=False)

In [ ]:
df_viviendas.head(10)

,index,Fecha extraccion,floor,price,propertyType,operation,size,exterior,rooms,bathrooms,...,hasLift,topNewDevelopment,isParkingSpaceIncludedInPrice,ParkingSpacePrice,distancia_minima_banco,banco_mas_cercano,numero_bancos_cercanos,distancia_minima_starbucks,starbucks_mas_cercano,numero_starbucks_cercanos
0,1,2382024,3,1500,flat,rent,80.0,1.0,3,2,...,True,False,NaN,NaN,15158.966797,bank,0,17677.365532,Starbucks,0
1,2,2382024,1,1496,flat,rent,55.0,NaN,2,2,...,True,False,1.0,NaN,336.168215,bank,8,1802.261546,Starbucks,0
2,3,2382024,2,1240,flat,rent,66.0,1.0,1,1,...,False,False,NaN,NaN,224.807509,bank,5,1276.570030,Starbucks,0
3,4,2382024,1,750,studio,rent,30.0,0.0,0,1,...,False,False,NaN,NaN,353.780818,bank,5,1939.304406,Starbucks,0
4,5,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,False,False,NaN,NaN,194.327806,bank,7,402.651194,Starbucks,1
5,6,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,False,False,NaN,NaN,194.327806,bank,7,402.651194,Starbucks,1
6,7,2382024,NaN,1050,studio,rent,15.0,NaN,0,1,...,False,False,NaN,NaN,86.257947,bank,19,147.711364,Starbucks,2
7,8,2382024,1,1275,flat,rent,39.0,NaN,1,1,...,False,False,NaN,NaN,186.870914,bank,8,552.199448,Starbucks,0
8,9,2382024,3,622,flat,rent,44.0,1.0,1,1,...,False,False,NaN,NaN,183.195953,bank,4,1596.041812,Starbucks,0
9,10,2382024,2,1500,studio,rent,35.0,1.0,0,1,...,False,False,NaN,NaN,232.317274,bank,4,610.229474,Starbucks,0


## Atractivos

In [ ]:
# Cargar el archivo CSV
spreadsheet_url = '/content/drive/MyDrive/TFM/Fuentes/bcn/bcn_atractivos.csv'
df_atractivos = pd.read_csv(spreadsheet_url)

df_atractivos.head()

,type,longitude,latitude
0,tourism,2.166505,41.388542
1,tourism,2.097302,41.360993
2,tourism,2.212445,41.428146
3,amenity,2.173416,41.380148
4,tourism,2.126851,41.409563


In [ ]:
# Añadir una columna para la distancia mínima y el banco más cercano
df_viviendas['distancia_minima_atractivos'] = float('inf')
df_viviendas['atractivos_mas_cercano'] = None

#Número de bancos cercanos:
df_viviendas['numero_atractivos_cercanos'] = 0

# Calcular la distancia geodésica de cada vivienda a cada banco
for i, vivienda in df_viviendas.iterrows():
    lat_viv, lon_viv = vivienda['latitude'], vivienda['longitude']
    distancias = df_atractivos.apply(lambda atractivos: geodesic((lat_viv, lon_viv), (atractivos['latitude'], atractivos['longitude'])).meters, axis=1)


#Encontrar el banco mas cercano:
    idx_min = distancias.idxmin()
    df_viviendas.at[i, 'distancia_minima_atractivos'] = distancias.min()
    df_viviendas.at[i, 'atractivos_mas_cercano'] = df_atractivos.at[idx_min, 'type']  # Ajusta según el nombre de la columna correspondiente

# Contar el número de bancos a menos de 500 metros
    num_atractivos_cercanos = (distancias <= 500).sum()
    df_viviendas.at[i, 'numero_atractivos_cercanos'] = num_atractivos_cercanos


# Guardar el archivo actualizado
df_viviendas.to_excel('/content/drive/MyDrive/TFM/Fuentes/Procesados/BCN_Prep_Data_23_08_2024.xlsx', index=False)


In [ ]:
df_viviendas.head(10)

,index,Fecha extraccion,floor,price,propertyType,operation,size,exterior,rooms,bathrooms,...,ParkingSpacePrice,distancia_minima_banco,banco_mas_cercano,numero_bancos_cercanos,distancia_minima_starbucks,starbucks_mas_cercano,numero_starbucks_cercanos,distancia_minima_atractivos,atractivos_mas_cercano,numero_atractivos_cercanos
0,1,2382024,3,1500,flat,rent,80.0,1.0,3,2,...,NaN,15158.966797,bank,0,17677.365532,Starbucks,0,14445.388325,tourism,0
1,2,2382024,1,1496,flat,rent,55.0,NaN,2,2,...,NaN,336.168215,bank,8,1802.261546,Starbucks,0,1034.426354,tourism,0
2,3,2382024,2,1240,flat,rent,66.0,1.0,1,1,...,NaN,224.807509,bank,5,1276.570030,Starbucks,0,1035.725133,tourism,0
3,4,2382024,1,750,studio,rent,30.0,0.0,0,1,...,NaN,353.780818,bank,5,1939.304406,Starbucks,0,1201.229548,tourism,0
4,5,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,NaN,194.327806,bank,7,402.651194,Starbucks,1,110.075500,tourism,21
5,6,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,NaN,194.327806,bank,7,402.651194,Starbucks,1,110.075500,tourism,21
6,7,2382024,NaN,1050,studio,rent,15.0,NaN,0,1,...,NaN,86.257947,bank,19,147.711364,Starbucks,2,62.819315,tourism,30
7,8,2382024,1,1275,flat,rent,39.0,NaN,1,1,...,NaN,186.870914,bank,8,552.199448,Starbucks,0,107.291828,tourism,12
8,9,2382024,3,622,flat,rent,44.0,1.0,1,1,...,NaN,183.195953,bank,4,1596.041812,Starbucks,0,701.713610,tourism,0
9,10,2382024,2,1500,studio,rent,35.0,1.0,0,1,...,NaN,232.317274,bank,4,610.229474,Starbucks,0,535.742679,tourism,0


## Buenasmigas

In [ ]:
# Cargar el archivo CSV
spreadsheet_url = '/content/drive/MyDrive/TFM/Fuentes/bcn/bcn_buenasmigas.csv'
df_buenasmigas = pd.read_csv(spreadsheet_url)

df_buenasmigas.head()

,name,longitude,latitude
0,Buenas Migas,2.135950,41.388900
1,Buenas Migas,2.174871,41.389928
2,Buenas Migas,2.151955,41.388713
3,Buenas Migas,2.148525,41.375556
4,Buenas Migas,2.177313,41.383808


In [ ]:
# Añadir una columna para la distancia mínima y el banco más cercano
df_viviendas['distancia_minima_buenasmigas'] = float('inf')
df_viviendas['buenasmigas_mas_cercano'] = None

#Número de bancos cercanos:
df_viviendas['numero_buenasmigas_cercanos'] = 0

# Calcular la distancia geodésica de cada vivienda a cada banco
for i, vivienda in df_viviendas.iterrows():
    lat_viv, lon_viv = vivienda['latitude'], vivienda['longitude']
    distancias = df_buenasmigas.apply(lambda buenasmigas: geodesic((lat_viv, lon_viv), (buenasmigas['latitude'], buenasmigas['longitude'])).meters, axis=1)


#Encontrar el banco mas cercano:
    idx_min = distancias.idxmin()
    df_viviendas.at[i, 'distancia_minima_buenasmigas'] = distancias.min()
    df_viviendas.at[i, 'buenasmigas_mas_cercano'] = df_buenasmigas.at[idx_min, 'name']  # Ajusta según el nombre de la columna correspondiente

# Contar el número de bancos a menos de 500 metros
    num_buenasmigas_cercanos = (distancias <= 500).sum()
    df_viviendas.at[i, 'numero_buenasmigas_cercanos'] = num_buenasmigas_cercanos


# Guardar el archivo actualizado
df_viviendas.to_excel('/content/drive/MyDrive/TFM/Fuentes/Procesados/BCN_Prep_Data_23_08_2024.xlsx', index=False)

In [ ]:
df_viviendas.head(10)

,index,Fecha extraccion,floor,price,propertyType,operation,size,exterior,rooms,bathrooms,...,numero_bancos_cercanos,distancia_minima_starbucks,starbucks_mas_cercano,numero_starbucks_cercanos,distancia_minima_atractivos,atractivos_mas_cercano,numero_atractivos_cercanos,distancia_minima_buenasmigas,buenasmigas_mas_cercano,numero_buenasmigas_cercanos
0,1,2382024,3,1500,flat,rent,80.0,1.0,3,2,...,0,17677.365532,Starbucks,0,14445.388325,tourism,0,19937.742710,Buenas Migas,0
1,2,2382024,1,1496,flat,rent,55.0,NaN,2,2,...,8,1802.261546,Starbucks,0,1034.426354,tourism,0,2358.385055,Buenas Migas,0
2,3,2382024,2,1240,flat,rent,66.0,1.0,1,1,...,5,1276.570030,Starbucks,0,1035.725133,tourism,0,1498.390746,Buenas Migas,0
3,4,2382024,1,750,studio,rent,30.0,0.0,0,1,...,5,1939.304406,Starbucks,0,1201.229548,tourism,0,2242.667683,Buenas Migas,0
4,5,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,7,402.651194,Starbucks,1,110.075500,tourism,21,525.177904,Buenas Migas,0
5,6,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,7,402.651194,Starbucks,1,110.075500,tourism,21,525.177904,Buenas Migas,0
6,7,2382024,NaN,1050,studio,rent,15.0,NaN,0,1,...,19,147.711364,Starbucks,2,62.819315,tourism,30,74.952110,Buenas Migas,1
7,8,2382024,1,1275,flat,rent,39.0,NaN,1,1,...,8,552.199448,Starbucks,0,107.291828,tourism,12,707.379759,Buenas Migas,0
8,9,2382024,3,622,flat,rent,44.0,1.0,1,1,...,4,1596.041812,Starbucks,0,701.713610,tourism,0,3740.115077,Buenas Migas,0
9,10,2382024,2,1500,studio,rent,35.0,1.0,0,1,...,4,610.229474,Starbucks,0,535.742679,tourism,0,140.444739,Buenas Migas,1


## Clubes nocturnos

In [ ]:
# Cargar el archivo CSV
spreadsheet_url = '/content/drive/MyDrive/TFM/Fuentes/bcn/bcn_clubesnocturnos.csv'
df_clubesnocturnos = pd.read_csv(spreadsheet_url)

df_clubesnocturnos.head()

,amenity,longitude,latitude
0,nightclub,2.152188,41.395476
1,nightclub,2.173943,41.389569
2,nightclub,2.094611,41.355506
3,nightclub,2.107681,41.362110
4,nightclub,2.150048,41.391894


In [ ]:
# Añadir una columna para la distancia mínima y el banco más cercano
df_viviendas['distancia_minima_clubesnocturnos'] = float('inf')
df_viviendas['clubesnocturnos_mas_cercano'] = None

#Número de bancos cercanos:
df_viviendas['numero_clubesnocturnos_cercanos'] = 0

# Calcular la distancia geodésica de cada vivienda a cada banco
for i, vivienda in df_viviendas.iterrows():
    lat_viv, lon_viv = vivienda['latitude'], vivienda['longitude']
    distancias = df_clubesnocturnos.apply(lambda clubesnocturnos: geodesic((lat_viv, lon_viv), (clubesnocturnos['latitude'], clubesnocturnos['longitude'])).meters, axis=1)


#Encontrar el banco mas cercano:
    idx_min = distancias.idxmin()
    df_viviendas.at[i, 'distancia_minima_clubesnocturnos'] = distancias.min()
    df_viviendas.at[i, 'clubesnocturnos_mas_cercano'] = df_clubesnocturnos.at[idx_min, 'amenity']  # Ajusta según el nombre de la columna correspondiente

# Contar el número de bancos a menos de 500 metros
    num_clubesnocturnos_cercanos = (distancias <= 500).sum()
    df_viviendas.at[i, 'numero_clubesnocturnos_cercanos'] = num_clubesnocturnos_cercanos


# Guardar el archivo actualizado
df_viviendas.to_excel('/content/drive/MyDrive/TFM/Fuentes/Procesados/BCN_Prep_Data_23_08_2024.xlsx', index=False)


In [ ]:
df_viviendas.head(10)

,index,Fecha extraccion,floor,price,propertyType,operation,size,exterior,rooms,bathrooms,...,numero_starbucks_cercanos,distancia_minima_atractivos,atractivos_mas_cercano,numero_atractivos_cercanos,distancia_minima_buenasmigas,buenasmigas_mas_cercano,numero_buenasmigas_cercanos,distancia_minima_clubesnocturnos,clubesnocturnos_mas_cercano,numero_clubesnocturnos_cercanos
0,1,2382024,3,1500,flat,rent,80.0,1.0,3,2,...,0,14445.388325,tourism,0,19937.742710,Buenas Migas,0,17954.809463,nightclub,0
1,2,2382024,1,1496,flat,rent,55.0,NaN,2,2,...,0,1034.426354,tourism,0,2358.385055,Buenas Migas,0,636.824667,nightclub,0
2,3,2382024,2,1240,flat,rent,66.0,1.0,1,1,...,0,1035.725133,tourism,0,1498.390746,Buenas Migas,0,1514.623589,nightclub,0
3,4,2382024,1,750,studio,rent,30.0,0.0,0,1,...,0,1201.229548,tourism,0,2242.667683,Buenas Migas,0,867.423997,nightclub,0
4,5,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,1,110.075500,tourism,21,525.177904,Buenas Migas,0,594.078017,nightclub,0
5,6,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,1,110.075500,tourism,21,525.177904,Buenas Migas,0,594.078017,nightclub,0
6,7,2382024,NaN,1050,studio,rent,15.0,NaN,0,1,...,2,62.819315,tourism,30,74.952110,Buenas Migas,1,251.052019,nightclub,4
7,8,2382024,1,1275,flat,rent,39.0,NaN,1,1,...,0,107.291828,tourism,12,707.379759,Buenas Migas,0,119.987893,nightclub,6
8,9,2382024,3,622,flat,rent,44.0,1.0,1,1,...,0,701.713610,tourism,0,3740.115077,Buenas Migas,0,2521.292637,nightclub,0
9,10,2382024,2,1500,studio,rent,35.0,1.0,0,1,...,0,535.742679,tourism,0,140.444739,Buenas Migas,1,171.350017,nightclub,2


## McDonalds

In [ ]:
# Cargar el archivo CSV
spreadsheet_url = '/content/drive/MyDrive/TFM/Fuentes/bcn/bcn_mcdonalds.csv'
df_mcdonalds = pd.read_csv(spreadsheet_url)

df_mcdonalds.head()

,name,longitude,latitude
0,McDonald's,2.173364,41.403817
1,McDonald's,2.185340,41.382600
2,McDonald's,2.151233,41.389222
3,McDonald's,2.183269,41.375554
4,McDonald's,2.120313,41.375822


In [ ]:
# Añadir una columna para la distancia mínima y el banco más cercano
df_viviendas['distancia_minima_mcdonalds'] = float('inf')
df_viviendas['mcdonalds_mas_cercano'] = None

#Número de bancos cercanos:
df_viviendas['numero_mcdonalds_cercanos'] = 0

# Calcular la distancia geodésica de cada vivienda a cada banco
for i, vivienda in df_viviendas.iterrows():
    lat_viv, lon_viv = vivienda['latitude'], vivienda['longitude']
    distancias = df_mcdonalds.apply(lambda mcdonalds: geodesic((lat_viv, lon_viv), (mcdonalds['latitude'], mcdonalds['longitude'])).meters, axis=1)


#Encontrar el banco mas cercano:
    idx_min = distancias.idxmin()
    df_viviendas.at[i, 'distancia_minima_mcdonalds'] = distancias.min()
    df_viviendas.at[i, 'mcdonalds_mas_cercano'] = df_mcdonalds.at[idx_min, 'name']  # Ajusta según el nombre de la columna correspondiente

# Contar el número de bancos a menos de 500 metros
    num_mcdonalds_cercanos = (distancias <= 500).sum()
    df_viviendas.at[i, 'numero_mcdonalds_cercanos'] = num_mcdonalds_cercanos


# Guardar el archivo actualizado
df_viviendas.to_excel('/content/drive/MyDrive/TFM/Fuentes/Procesados/BCN_Prep_Data_23_08_2024.xlsx', index=False)


In [ ]:
df_viviendas.head(10)

,index,Fecha extraccion,floor,price,propertyType,operation,size,exterior,rooms,bathrooms,...,numero_atractivos_cercanos,distancia_minima_buenasmigas,buenasmigas_mas_cercano,numero_buenasmigas_cercanos,distancia_minima_clubesnocturnos,clubesnocturnos_mas_cercano,numero_clubesnocturnos_cercanos,distancia_minima_mcdonalds,mcdonalds_mas_cercano,numero_mcdonalds_cercanos
0,1,2382024,3,1500,flat,rent,80.0,1.0,3,2,...,0,19937.742710,Buenas Migas,0,17954.809463,nightclub,0,14731.496860,McDonald's,0
1,2,2382024,1,1496,flat,rent,55.0,NaN,2,2,...,0,2358.385055,Buenas Migas,0,636.824667,nightclub,0,541.492641,McDonald's,0
2,3,2382024,2,1240,flat,rent,66.0,1.0,1,1,...,0,1498.390746,Buenas Migas,0,1514.623589,nightclub,0,1278.961851,McDonald's,0
3,4,2382024,1,750,studio,rent,30.0,0.0,0,1,...,0,2242.667683,Buenas Migas,0,867.423997,nightclub,0,726.725769,McDonald's,0
4,5,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,21,525.177904,Buenas Migas,0,594.078017,nightclub,0,699.208251,McDonald's,0
5,6,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,21,525.177904,Buenas Migas,0,594.078017,nightclub,0,699.208251,McDonald's,0
6,7,2382024,NaN,1050,studio,rent,15.0,NaN,0,1,...,30,74.952110,Buenas Migas,1,251.052019,nightclub,4,452.670989,McDonald's,1
7,8,2382024,1,1275,flat,rent,39.0,NaN,1,1,...,12,707.379759,Buenas Migas,0,119.987893,nightclub,6,288.400734,McDonald's,1
8,9,2382024,3,622,flat,rent,44.0,1.0,1,1,...,0,3740.115077,Buenas Migas,0,2521.292637,nightclub,0,629.690124,McDonald's,0
9,10,2382024,2,1500,studio,rent,35.0,1.0,0,1,...,0,140.444739,Buenas Migas,1,171.350017,nightclub,2,536.026861,McDonald's,0


## Metro

In [ ]:
# Cargar el archivo CSV
spreadsheet_url = '/content/drive/MyDrive/TFM/Fuentes/bcn/bcn_metro.csv'
df_metro = pd.read_csv(spreadsheet_url)

df_metro.head()

,type,longitude,latitude
0,metro_station,2.131104,41.416459
1,metro_station,2.119497,41.421382
2,metro_station,2.108227,41.400565
3,metro_station,2.133331,41.372795
4,metro_station,2.135323,41.375630


In [ ]:
# Añadir una columna para la distancia mínima y el banco más cercano
df_viviendas['distancia_minima_metro'] = float('inf')
df_viviendas['metro_mas_cercano'] = None

#Número de bancos cercanos:
df_viviendas['numero_metro_cercanos'] = 0

# Calcular la distancia geodésica de cada vivienda a cada banco
for i, vivienda in df_viviendas.iterrows():
    lat_viv, lon_viv = vivienda['latitude'], vivienda['longitude']
    distancias = df_metro.apply(lambda metro: geodesic((lat_viv, lon_viv), (metro['latitude'], metro['longitude'])).meters, axis=1)


#Encontrar el banco mas cercano:
    idx_min = distancias.idxmin()
    df_viviendas.at[i, 'distancia_minima_metro'] = distancias.min()
    df_viviendas.at[i, 'metro_mas_cercano'] = df_metro.at[idx_min, 'type']  # Ajusta según el nombre de la columna correspondiente

# Contar el número de bancos a menos de 500 metros
    num_metro_cercanos = (distancias <= 500).sum()
    df_viviendas.at[i, 'numero_metro_cercanos'] = num_metro_cercanos


# Guardar el archivo actualizado
df_viviendas.to_excel('/content/drive/MyDrive/TFM/Fuentes/Procesados/BCN_Prep_Data_23_08_2024.xlsx', index=False)


In [ ]:
df_viviendas.head(10)

,index,Fecha extraccion,floor,price,propertyType,operation,size,exterior,rooms,bathrooms,...,numero_buenasmigas_cercanos,distancia_minima_clubesnocturnos,clubesnocturnos_mas_cercano,numero_clubesnocturnos_cercanos,distancia_minima_mcdonalds,mcdonalds_mas_cercano,numero_mcdonalds_cercanos,distancia_minima_metro,metro_mas_cercano,numero_metro_cercanos
0,1,2382024,3,1500,flat,rent,80.0,1.0,3,2,...,0,17954.809463,nightclub,0,14731.496860,McDonald's,0,15498.965729,metro_station,0
1,2,2382024,1,1496,flat,rent,55.0,NaN,2,2,...,0,636.824667,nightclub,0,541.492641,McDonald's,0,453.680860,metro_station,1
2,3,2382024,2,1240,flat,rent,66.0,1.0,1,1,...,0,1514.623589,nightclub,0,1278.961851,McDonald's,0,80.197196,metro_station,2
3,4,2382024,1,750,studio,rent,30.0,0.0,0,1,...,0,867.423997,nightclub,0,726.725769,McDonald's,0,483.556903,metro_station,1
4,5,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,0,594.078017,nightclub,0,699.208251,McDonald's,0,453.540110,metro_station,2
5,6,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,0,594.078017,nightclub,0,699.208251,McDonald's,0,453.540110,metro_station,2
6,7,2382024,NaN,1050,studio,rent,15.0,NaN,0,1,...,1,251.052019,nightclub,4,452.670989,McDonald's,1,93.617162,metro_station,2
7,8,2382024,1,1275,flat,rent,39.0,NaN,1,1,...,0,119.987893,nightclub,6,288.400734,McDonald's,1,174.879233,metro_station,2
8,9,2382024,3,622,flat,rent,44.0,1.0,1,1,...,0,2521.292637,nightclub,0,629.690124,McDonald's,0,212.450719,metro_station,1
9,10,2382024,2,1500,studio,rent,35.0,1.0,0,1,...,1,171.350017,nightclub,2,536.026861,McDonald's,0,699.326772,metro_station,0


## Parking

In [ ]:
import pandas as pd

# Ruta al archivo Excel en Google Drive

spreadsheet_url = '/content/drive/MyDrive/TFM/Fuentes/Procesados/BCN_Prep_Data_23_08_2024.xlsx'
# Leer el archivo Excel
df_viviendas = pd.read_excel(spreadsheet_url)

# Mostrar las primeras filas del DataFrame
df_viviendas.head()

,index,Fecha extraccion,floor,price,propertyType,operation,size,exterior,rooms,bathrooms,...,numero_buenasmigas_cercanos,distancia_minima_clubesnocturnos,clubesnocturnos_mas_cercano,numero_clubesnocturnos_cercanos,distancia_minima_mcdonalds,mcdonalds_mas_cercano,numero_mcdonalds_cercanos,distancia_minima_metro,metro_mas_cercano,numero_metro_cercanos
0,1,2382024,3,1500,flat,rent,80.0,1.0,3,2,...,0,17954.809463,nightclub,0,14731.496860,McDonald's,0,15498.965729,metro_station,0
1,2,2382024,1,1496,flat,rent,55.0,NaN,2,2,...,0,636.824667,nightclub,0,541.492641,McDonald's,0,453.680860,metro_station,1
2,3,2382024,2,1240,flat,rent,66.0,1.0,1,1,...,0,1514.623589,nightclub,0,1278.961851,McDonald's,0,80.197196,metro_station,2
3,4,2382024,1,750,studio,rent,30.0,0.0,0,1,...,0,867.423997,nightclub,0,726.725769,McDonald's,0,483.556903,metro_station,1
4,5,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,0,594.078017,nightclub,0,699.208251,McDonald's,0,453.540110,metro_station,2


In [ ]:
# Cargar el archivo CSV
spreadsheet_url = '/content/drive/MyDrive/TFM/Fuentes/bcn/bcn_parking.csv'
df_parking = pd.read_csv(spreadsheet_url)

df_parking.head()

,amenity,longitude,latitude
0,parking,2.151751,41.384467
1,parking,2.151180,41.393839
2,parking,2.150554,41.393464
3,parking,2.194204,41.461010
4,parking,2.194076,41.460956


In [ ]:
# Añadir una columna para la distancia mínima y el banco más cercano
df_viviendas['distancia_minima_parking'] = float('inf')
df_viviendas['parking_mas_cercano'] = None

#Número de bancos cercanos:
df_viviendas['numero_parking_cercanos'] = 0

# Calcular la distancia geodésica de cada vivienda a cada banco
for i, vivienda in df_viviendas.iterrows():
    lat_viv, lon_viv = vivienda['latitude'], vivienda['longitude']
    distancias = df_parking.apply(lambda parking: geodesic((lat_viv, lon_viv), (parking['latitude'], parking['longitude'])).meters, axis=1)


#Encontrar el banco mas cercano:
    idx_min = distancias.idxmin()
    df_viviendas.at[i, 'distancia_minima_parking'] = distancias.min()
    df_viviendas.at[i, 'parking_mas_cercano'] = df_parking.at[idx_min, 'amenity']  # Ajusta según el nombre de la columna correspondiente

# Contar el número de bancos a menos de 500 parkings
    num_parking_cercanos = (distancias <= 500).sum()
    df_viviendas.at[i, 'numero_parking_cercanos'] = num_parking_cercanos


# Guardar el archivo actualizado
df_viviendas.to_excel('/content/drive/MyDrive/TFM/Fuentes/Procesados/BCN_Prep_Data_23_08_2024.xlsx', index=False)


In [ ]:
df_viviendas.head(10)

,index,Fecha extraccion,floor,price,propertyType,operation,size,exterior,rooms,bathrooms,...,numero_clubesnocturnos_cercanos,distancia_minima_mcdonalds,mcdonalds_mas_cercano,numero_mcdonalds_cercanos,distancia_minima_metro,metro_mas_cercano,numero_metro_cercanos,distancia_minima_parking,parking_mas_cercano,numero_parking_cercanos
0,1,2382024,3,1500,flat,rent,80.0,1.0,3,2,...,0,14731.496860,McDonald's,0,15498.965729,metro_station,0,14681.930902,parking,0
1,2,2382024,1,1496,flat,rent,55.0,NaN,2,2,...,0,541.492641,McDonald's,0,453.680860,metro_station,1,411.736065,parking,2
2,3,2382024,2,1240,flat,rent,66.0,1.0,1,1,...,0,1278.961851,McDonald's,0,80.197196,metro_station,2,106.145484,parking,8
3,4,2382024,1,750,studio,rent,30.0,0.0,0,1,...,0,726.725769,McDonald's,0,483.556903,metro_station,1,285.163443,parking,7
4,5,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,0,699.208251,McDonald's,0,453.540110,metro_station,2,76.146885,parking,9
5,6,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,0,699.208251,McDonald's,0,453.540110,metro_station,2,76.146885,parking,9
6,7,2382024,NaN,1050,studio,rent,15.0,NaN,0,1,...,4,452.670989,McDonald's,1,93.617162,metro_station,2,228.450229,parking,3
7,8,2382024,1,1275,flat,rent,39.0,NaN,1,1,...,6,288.400734,McDonald's,1,174.879233,metro_station,2,122.069410,parking,9
8,9,2382024,3,622,flat,rent,44.0,1.0,1,1,...,0,629.690124,McDonald's,0,212.450719,metro_station,1,160.579439,parking,8
9,10,2382024,2,1500,studio,rent,35.0,1.0,0,1,...,2,536.026861,McDonald's,0,699.326772,metro_station,0,185.032564,parking,7


## Parkingbici

In [ ]:
# Cargar el archivo CSV
spreadsheet_url = '/content/drive/MyDrive/TFM/Fuentes/bcn/bcn_parkingbici.csv'
df_parkingbici = pd.read_csv(spreadsheet_url)

df_parkingbici.head()

,amenity,longitude,latitude
0,bicycle_parking,2.151532,41.394208
1,bicycle_parking,2.154054,41.385862
2,bicycle_parking,2.154853,41.386524
3,bicycle_parking,2.167195,41.380377
4,bicycle_parking,2.157520,41.395702


In [ ]:
# Añadir una columna para la distancia mínima y el banco más cercano
df_viviendas['distancia_minima_parkingbici'] = float('inf')
df_viviendas['parkingbici_mas_cercano'] = None

#Número de bancos cercanos:
df_viviendas['numero_parkingbici_cercanos'] = 0

# Calcular la distancia geodésica de cada vivienda a cada banco
for i, vivienda in df_viviendas.iterrows():
    lat_viv, lon_viv = vivienda['latitude'], vivienda['longitude']
    distancias = df_parkingbici.apply(lambda parkingbici: geodesic((lat_viv, lon_viv), (parkingbici['latitude'], parkingbici['longitude'])).meters, axis=1)


#Encontrar el banco mas cercano:
    idx_min = distancias.idxmin()
    df_viviendas.at[i, 'distancia_minima_parkingbici'] = distancias.min()
    df_viviendas.at[i, 'parkingbici_mas_cercano'] = df_parkingbici.at[idx_min, 'amenity']  # Ajusta según el nombre de la columna correspondiente

# Contar el número de bancos a menos de 500 parkingbicis
    num_parkingbici_cercanos = (distancias <= 500).sum()
    df_viviendas.at[i, 'numero_parkingbici_cercanos'] = num_parkingbici_cercanos


# Guardar el archivo actualizado
df_viviendas.to_excel('/content/drive/MyDrive/TFM/Fuentes/Procesados/BCN_Prep_Data_23_08_2024.xlsx', index=False)


In [ ]:
df_viviendas.head(10)

,index,Fecha extraccion,floor,price,propertyType,operation,size,exterior,rooms,bathrooms,...,numero_mcdonalds_cercanos,distancia_minima_metro,metro_mas_cercano,numero_metro_cercanos,distancia_minima_parking,parking_mas_cercano,numero_parking_cercanos,distancia_minima_parkingbici,parkingbici_mas_cercano,numero_parkingbici_cercanos
0,1,2382024,3,1500,flat,rent,80.0,1.0,3,2,...,0,15498.965729,metro_station,0,14681.930902,parking,0,15735.041227,bicycle_parking,0
1,2,2382024,1,1496,flat,rent,55.0,NaN,2,2,...,0,453.680860,metro_station,1,411.736065,parking,2,49.474047,bicycle_parking,24
2,3,2382024,2,1240,flat,rent,66.0,1.0,1,1,...,0,80.197196,metro_station,2,106.145484,parking,8,50.429191,bicycle_parking,20
3,4,2382024,1,750,studio,rent,30.0,0.0,0,1,...,0,483.556903,metro_station,1,285.163443,parking,7,17.429113,bicycle_parking,21
4,5,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,0,453.540110,metro_station,2,76.146885,parking,9,18.436421,bicycle_parking,107
5,6,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,0,453.540110,metro_station,2,76.146885,parking,9,18.436421,bicycle_parking,107
6,7,2382024,NaN,1050,studio,rent,15.0,NaN,0,1,...,1,93.617162,metro_station,2,228.450229,parking,3,51.814785,bicycle_parking,84
7,8,2382024,1,1275,flat,rent,39.0,NaN,1,1,...,1,174.879233,metro_station,2,122.069410,parking,9,88.227295,bicycle_parking,72
8,9,2382024,3,622,flat,rent,44.0,1.0,1,1,...,0,212.450719,metro_station,1,160.579439,parking,8,316.124556,bicycle_parking,2
9,10,2382024,2,1500,studio,rent,35.0,1.0,0,1,...,0,699.326772,metro_station,0,185.032564,parking,7,62.248911,bicycle_parking,43


## Parques

In [ ]:
# Cargar el archivo CSV
spreadsheet_url = '/content/drive/MyDrive/TFM/Fuentes/bcn/bcn_parques.csv'
df_parques = pd.read_csv(spreadsheet_url)

df_parques.head()

,type,longitude,latitude
0,park,2.215169,41.411868
1,park,2.208998,41.426122
2,park,2.138652,41.372389
3,park,2.141240,41.371221
4,park,2.195266,41.449076


In [ ]:
# Añadir una columna para la distancia mínima y el banco más cercano
df_viviendas['distancia_minima_parques'] = float('inf')
df_viviendas['parques_mas_cercano'] = None

#Número de bancos cercanos:
df_viviendas['numero_parques_cercanos'] = 0

# Calcular la distancia geodésica de cada vivienda a cada banco
for i, vivienda in df_viviendas.iterrows():
    lat_viv, lon_viv = vivienda['latitude'], vivienda['longitude']
    distancias = df_parques.apply(lambda parques: geodesic((lat_viv, lon_viv), (parques['latitude'], parques['longitude'])).meters, axis=1)


#Encontrar el banco mas cercano:
    idx_min = distancias.idxmin()
    df_viviendas.at[i, 'distancia_minima_parques'] = distancias.min()
    df_viviendas.at[i, 'parques_mas_cercano'] = df_parques.at[idx_min, 'type']  # Ajusta según el nombre de la columna correspondiente

# Contar el número de bancos a menos de 500 parquess
    num_parques_cercanos = (distancias <= 500).sum()
    df_viviendas.at[i, 'numero_parques_cercanos'] = num_parques_cercanos


# Guardar el archivo actualizado
df_viviendas.to_excel('/content/drive/MyDrive/TFM/Fuentes/Procesados/BCN_Prep_Data_23_08_2024.xlsx', index=False)


In [ ]:
df_viviendas.head(10)

,index,Fecha extraccion,floor,price,propertyType,operation,size,exterior,rooms,bathrooms,...,numero_metro_cercanos,distancia_minima_parking,parking_mas_cercano,numero_parking_cercanos,distancia_minima_parkingbici,parkingbici_mas_cercano,numero_parkingbici_cercanos,distancia_minima_parques,parques_mas_cercano,numero_parques_cercanos
0,1,2382024,3,1500,flat,rent,80.0,1.0,3,2,...,0,14681.930902,parking,0,15735.041227,bicycle_parking,0,15011.966357,park,0
1,2,2382024,1,1496,flat,rent,55.0,NaN,2,2,...,1,411.736065,parking,2,49.474047,bicycle_parking,24,19.806682,park,9
2,3,2382024,2,1240,flat,rent,66.0,1.0,1,1,...,2,106.145484,parking,8,50.429191,bicycle_parking,20,131.594516,park,13
3,4,2382024,1,750,studio,rent,30.0,0.0,0,1,...,1,285.163443,parking,7,17.429113,bicycle_parking,21,163.894902,park,7
4,5,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,2,76.146885,parking,9,18.436421,bicycle_parking,107,231.329522,park,1
5,6,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,2,76.146885,parking,9,18.436421,bicycle_parking,107,231.329522,park,1
6,7,2382024,NaN,1050,studio,rent,15.0,NaN,0,1,...,2,228.450229,parking,3,51.814785,bicycle_parking,84,212.586323,park,4
7,8,2382024,1,1275,flat,rent,39.0,NaN,1,1,...,2,122.069410,parking,9,88.227295,bicycle_parking,72,384.564206,park,4
8,9,2382024,3,622,flat,rent,44.0,1.0,1,1,...,1,160.579439,parking,8,316.124556,bicycle_parking,2,118.257111,park,4
9,10,2382024,2,1500,studio,rent,35.0,1.0,0,1,...,0,185.032564,parking,7,62.248911,bicycle_parking,43,585.788187,park,0


## Playas

In [ ]:
# Cargar el archivo CSV
spreadsheet_url = '/content/drive/MyDrive/TFM/Fuentes/bcn/bcn_playa.csv'
df_playas = pd.read_csv(spreadsheet_url)

df_playas.head()

,type,longitude,latitude
0,beach,2.219671,41.405475
1,beach,2.196277,41.383725
2,beach,2.213364,41.400289


In [ ]:
# Añadir una columna para la distancia mínima y el banco más cercano
df_viviendas['distancia_minima_playas'] = float('inf')
df_viviendas['playas_mas_cercano'] = None

#Número de bancos cercanos:
df_viviendas['numero_playas_cercanos'] = 0

# Calcular la distancia geodésica de cada vivienda a cada banco
for i, vivienda in df_viviendas.iterrows():
    lat_viv, lon_viv = vivienda['latitude'], vivienda['longitude']
    distancias = df_playas.apply(lambda playas: geodesic((lat_viv, lon_viv), (playas['latitude'], playas['longitude'])).meters, axis=1)


#Encontrar el banco mas cercano:
    idx_min = distancias.idxmin()
    df_viviendas.at[i, 'distancia_minima_playas'] = distancias.min()
    df_viviendas.at[i, 'playas_mas_cercano'] = df_playas.at[idx_min, 'type']  # Ajusta según el nombre de la columna correspondiente

# Contar el número de bancos a menos de 500 playass
    num_playas_cercanos = (distancias <= 500).sum()
    df_viviendas.at[i, 'numero_playas_cercanos'] = num_playas_cercanos


# Guardar el archivo actualizado
df_viviendas.to_excel('/content/drive/MyDrive/TFM/Fuentes/Procesados/BCN_Prep_Data_23_08_2024.xlsx', index=False)

In [ ]:
df_viviendas.head(10)

,index,Fecha extraccion,floor,price,propertyType,operation,size,exterior,rooms,bathrooms,...,numero_parking_cercanos,distancia_minima_parkingbici,parkingbici_mas_cercano,numero_parkingbici_cercanos,distancia_minima_parques,parques_mas_cercano,numero_parques_cercanos,distancia_minima_playas,playas_mas_cercano,numero_playas_cercanos
0,1,2382024,3,1500,flat,rent,80.0,1.0,3,2,...,0,15735.041227,bicycle_parking,0,15011.966357,park,0,18027.720852,beach,0
1,2,2382024,1,1496,flat,rent,55.0,NaN,2,2,...,2,49.474047,bicycle_parking,24,19.806682,park,9,6466.202042,beach,0
2,3,2382024,2,1240,flat,rent,66.0,1.0,1,1,...,8,50.429191,bicycle_parking,20,131.594516,park,13,6262.730747,beach,0
3,4,2382024,1,750,studio,rent,30.0,0.0,0,1,...,7,17.429113,bicycle_parking,21,163.894902,park,7,6334.575359,beach,0
4,5,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,9,18.436421,bicycle_parking,107,231.329522,park,1,1383.495769,beach,0
5,6,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,9,18.436421,bicycle_parking,107,231.329522,park,1,1383.495769,beach,0
6,7,2382024,NaN,1050,studio,rent,15.0,NaN,0,1,...,3,51.814785,bicycle_parking,84,212.586323,park,4,1530.855277,beach,0
7,8,2382024,1,1275,flat,rent,39.0,NaN,1,1,...,9,88.227295,bicycle_parking,72,384.564206,park,4,1950.956842,beach,0
8,9,2382024,3,622,flat,rent,44.0,1.0,1,1,...,8,316.124556,bicycle_parking,2,118.257111,park,4,3206.686070,beach,0
9,10,2382024,2,1500,studio,rent,35.0,1.0,0,1,...,7,62.248911,bicycle_parking,43,585.788187,park,0,971.852243,beach,0


## Santagloria

In [ ]:
# Cargar el archivo CSV
spreadsheet_url = '/content/drive/MyDrive/TFM/Fuentes/bcn/bcn_santagloria.csv'
df_santagloria = pd.read_csv(spreadsheet_url)

df_santagloria.head()

,name,longitude,latitude
0,Santagloria,2.121132,41.401475
1,Santagloria,2.133309,41.403549
2,Santagloria,2.123639,41.397084
3,Santagloria,2.168117,41.394837
4,Santagloria,2.124357,41.386393


In [ ]:
# Añadir una columna para la distancia mínima y el banco más cercano
df_viviendas['distancia_minima_santagloria'] = float('inf')
df_viviendas['santagloria_mas_cercano'] = None

#Número de bancos cercanos:
df_viviendas['numero_santagloria_cercanos'] = 0

# Calcular la distancia geodésica de cada vivienda a cada banco
for i, vivienda in df_viviendas.iterrows():
    lat_viv, lon_viv = vivienda['latitude'], vivienda['longitude']
    distancias = df_santagloria.apply(lambda santagloria: geodesic((lat_viv, lon_viv), (santagloria['latitude'], santagloria['longitude'])).meters, axis=1)


#Encontrar el banco mas cercano:
    idx_min = distancias.idxmin()
    df_viviendas.at[i, 'distancia_minima_santagloria'] = distancias.min()
    df_viviendas.at[i, 'santagloria_mas_cercano'] = df_santagloria.at[idx_min, 'name']  # Ajusta según el nombre de la columna correspondiente

# Contar el número de bancos a menos de 500 santaglorias
    num_santagloria_cercanos = (distancias <= 500).sum()
    df_viviendas.at[i, 'numero_santagloria_cercanos'] = num_santagloria_cercanos


# Guardar el archivo actualizado
df_viviendas.to_excel('/content/drive/MyDrive/TFM/Fuentes/Procesados/BCN_Prep_Data_23_08_2024.xlsx', index=False)

In [ ]:
df_viviendas.head(10)

,index,Fecha extraccion,floor,price,propertyType,operation,size,exterior,rooms,bathrooms,...,numero_parkingbici_cercanos,distancia_minima_parques,parques_mas_cercano,numero_parques_cercanos,distancia_minima_playas,playas_mas_cercano,numero_playas_cercanos,distancia_minima_santagloria,santagloria_mas_cercano,numero_santagloria_cercanos
0,1,2382024,3,1500,flat,rent,80.0,1.0,3,2,...,0,15011.966357,park,0,18027.720852,beach,0,21792.634319,Santagloria,0
1,2,2382024,1,1496,flat,rent,55.0,NaN,2,2,...,24,19.806682,park,9,6466.202042,beach,0,1738.390045,Santagloria,0
2,3,2382024,2,1240,flat,rent,66.0,1.0,1,1,...,20,131.594516,park,13,6262.730747,beach,0,230.295035,Santagloria,2
3,4,2382024,1,750,studio,rent,30.0,0.0,0,1,...,21,163.894902,park,7,6334.575359,beach,0,1871.152485,Santagloria,0
4,5,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,107,231.329522,park,1,1383.495769,beach,0,1306.797612,Santagloria,0
5,6,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,107,231.329522,park,1,1383.495769,beach,0,1306.797612,Santagloria,0
6,7,2382024,NaN,1050,studio,rent,15.0,NaN,0,1,...,84,212.586323,park,4,1530.855277,beach,0,1518.519854,Santagloria,0
7,8,2382024,1,1275,flat,rent,39.0,NaN,1,1,...,72,384.564206,park,4,1950.956842,beach,0,1955.110871,Santagloria,0
8,9,2382024,3,622,flat,rent,44.0,1.0,1,1,...,2,118.257111,park,4,3206.686070,beach,0,5325.243844,Santagloria,0
9,10,2382024,2,1500,studio,rent,35.0,1.0,0,1,...,43,585.788187,park,0,971.852243,beach,0,2702.962299,Santagloria,0


## Vivari

In [ ]:
# Cargar el archivo CSV
spreadsheet_url = '/content/drive/MyDrive/TFM/Fuentes/bcn/bcn_vivari.csv'
df_vivari = pd.read_csv(spreadsheet_url)

df_vivari.head()

,name,longitude,latitude
0,Vivari,2.141115,41.368661
1,Vivari,2.180505,41.394718
2,Vivari,2.181059,41.414915
3,Vivari,2.174203,41.425883
4,Vivari,2.172190,41.390292


In [ ]:
# Añadir una columna para la distancia mínima y el banco más cercano
df_viviendas['distancia_minima_vivari'] = float('inf')
df_viviendas['vivari_mas_cercano'] = None

#Número de bancos cercanos:
df_viviendas['numero_vivari_cercanos'] = 0

# Calcular la distancia geodésica de cada vivienda a cada banco
for i, vivienda in df_viviendas.iterrows():
    lat_viv, lon_viv = vivienda['latitude'], vivienda['longitude']
    distancias = df_vivari.apply(lambda vivari: geodesic((lat_viv, lon_viv), (vivari['latitude'], vivari['longitude'])).meters, axis=1)


#Encontrar el banco mas cercano:
    idx_min = distancias.idxmin()
    df_viviendas.at[i, 'distancia_minima_vivari'] = distancias.min()
    df_viviendas.at[i, 'vivari_mas_cercano'] = df_vivari.at[idx_min, 'name']  # Ajusta según el nombre de la columna correspondiente

# Contar el número de bancos a menos de 500 vivaris
    num_vivari_cercanos = (distancias <= 500).sum()
    df_viviendas.at[i, 'numero_vivari_cercanos'] = num_vivari_cercanos


# Guardar el archivo actualizado
df_viviendas.to_excel('/content/drive/MyDrive/TFM/Fuentes/Procesados/BCN_Prep_Data_23_08_2024.xlsx', index=False)


In [ ]:
df_viviendas.head(10)

,index,Fecha extraccion,floor,price,propertyType,operation,size,exterior,rooms,bathrooms,...,numero_parques_cercanos,distancia_minima_playas,playas_mas_cercano,numero_playas_cercanos,distancia_minima_santagloria,santagloria_mas_cercano,numero_santagloria_cercanos,distancia_minima_vivari,vivari_mas_cercano,numero_vivari_cercanos
0,1,2382024,3,1500,flat,rent,80.0,1.0,3,2,...,0,18027.720852,beach,0,21792.634319,Santagloria,0,15404.139470,Vivari,0
1,2,2382024,1,1496,flat,rent,55.0,NaN,2,2,...,9,6466.202042,beach,0,1738.390045,Santagloria,0,643.878222,Vivari,0
2,3,2382024,2,1240,flat,rent,66.0,1.0,1,1,...,13,6262.730747,beach,0,230.295035,Santagloria,2,1455.624298,Vivari,0
3,4,2382024,1,750,studio,rent,30.0,0.0,0,1,...,7,6334.575359,beach,0,1871.152485,Santagloria,0,484.047977,Vivari,1
4,5,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,1,1383.495769,beach,0,1306.797612,Santagloria,0,220.084909,Vivari,2
5,6,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,1,1383.495769,beach,0,1306.797612,Santagloria,0,220.084909,Vivari,2
6,7,2382024,NaN,1050,studio,rent,15.0,NaN,0,1,...,4,1530.855277,beach,0,1518.519854,Santagloria,0,369.591842,Vivari,1
7,8,2382024,1,1275,flat,rent,39.0,NaN,1,1,...,4,1950.956842,beach,0,1955.110871,Santagloria,0,890.316998,Vivari,0
8,9,2382024,3,622,flat,rent,44.0,1.0,1,1,...,4,3206.686070,beach,0,5325.243844,Santagloria,0,1248.209551,Vivari,0
9,10,2382024,2,1500,studio,rent,35.0,1.0,0,1,...,0,971.852243,beach,0,2702.962299,Santagloria,0,1447.940642,Vivari,0


# AIRBNBR número de airbnbs dentro de 500 metros

# Airbnb

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd


# Cargar el archivo CSV
spreadsheet_url = '/content/drive/MyDrive/TFM/Fuentes/Airbnb/airbnb_data_2.xlsx'
df_airbnb = pd.read_excel(spreadsheet_url)

df_airbnb.head()

# Filtrar NaN en las columnas latitude y longitude
df_airbnb = df_airbnb.dropna(subset=['latitude', 'longitude'])

**Procesado en paralelo**

In [ ]:
import pandas as pd

# Ruta al archivo Excel en Google Drive
spreadsheet_url = '/content/drive/MyDrive/TFM/Fuentes/Procesados/BCN_Prep_Data_23_08_2024.xlsx'


# Leer el archivo csv
df_viviendas = pd.read_excel(spreadsheet_url)

# Mostrar las primeras filas del DataFrame
df_viviendas.head()

,index,Fecha extraccion,floor,price,propertyType,operation,size,exterior,rooms,bathrooms,...,numero_parques_cercanos,distancia_minima_playas,playas_mas_cercano,numero_playas_cercanos,distancia_minima_santagloria,santagloria_mas_cercano,numero_santagloria_cercanos,distancia_minima_vivari,vivari_mas_cercano,numero_vivari_cercanos
0,1,2382024,3,1500,flat,rent,80.0,1.0,3,2,...,0,18027.720852,beach,0,21792.634319,Santagloria,0,15404.139470,Vivari,0
1,2,2382024,1,1496,flat,rent,55.0,NaN,2,2,...,9,6466.202042,beach,0,1738.390045,Santagloria,0,643.878222,Vivari,0
2,3,2382024,2,1240,flat,rent,66.0,1.0,1,1,...,13,6262.730747,beach,0,230.295035,Santagloria,2,1455.624298,Vivari,0
3,4,2382024,1,750,studio,rent,30.0,0.0,0,1,...,7,6334.575359,beach,0,1871.152485,Santagloria,0,484.047977,Vivari,1
4,5,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,1,1383.495769,beach,0,1306.797612,Santagloria,0,220.084909,Vivari,2


In [ ]:
from geopy.distance import geodesic
import pandas as pd
from multiprocessing import Pool, cpu_count

def contar_y_calcular_media_precios_wrapper(args):
    centro_lat, centro_lon, df_airbnb, radio_metros = args
    centro = (centro_lat, centro_lon)
    airbnbs_cercanos = df_airbnb[df_airbnb.apply(lambda row: geodesic(centro, (row['latitude'], row['longitude'])).meters <= radio_metros, axis=1)]
    num_airbnbs = len(airbnbs_cercanos)
    media_precios = airbnbs_cercanos['precio_euro'].mean() if num_airbnbs > 0 else 0
    return num_airbnbs, media_precios

def procesar_en_paralelo(df_viviendas, df_airbnb, radio_metros=500):
    args = [(row['latitude'], row['longitude'], df_airbnb, radio_metros) for idx, row in df_viviendas.iterrows()]

    # Usar todos los núcleos disponibles
    with Pool(cpu_count()) as pool:
        resultados = pool.map(contar_y_calcular_media_precios_wrapper, args)

    num_airbnbs, media_precios = zip(*resultados)
    df_viviendas['num_airbnbs_500'] = num_airbnbs
    df_viviendas['media_precios_airbnbs_500'] = media_precios

    return df_viviendas

# Supongamos que df_viviendas y df_airbnb ya están definidos y tienen las columnas 'latitude', 'longitude', y 'precio_euro'

# Procesar en paralelo
df_viviendas_procesado = procesar_en_paralelo(df_viviendas, df_airbnb)

# Guardar el archivo actualizado
archivo_salida = '/content/drive/MyDrive/TFM/Fuentes/Procesados/BCN_Prep_Data_23_08_2024.xlsx'
df_viviendas_procesado.to_excel(archivo_salida, index=False)

print(f"Proceso completado y archivo guardado correctamente en {archivo_salida}.")

Proceso completado y archivo guardado correctamente en /content/drive/MyDrive/TFM/Fuentes/Procesados/BCN_Prep_Data_23_08_2024.xlsx.


In [ ]:
import pandas as pd

# Ruta al archivo Excel en Google Drive

spreadsheet_url = '/content/drive/MyDrive/TFM/Fuentes/Procesados/BCN_Prep_Data_23_08_2024.xlsx'
# Leer el archivo Excel
df_viviendas = pd.read_excel(spreadsheet_url)


# Mostrar las primeras filas del DataFrame
df_viviendas.head()

,index,Fecha extraccion,floor,price,propertyType,operation,size,exterior,rooms,bathrooms,...,playas_mas_cercano,numero_playas_cercanos,distancia_minima_santagloria,santagloria_mas_cercano,numero_santagloria_cercanos,distancia_minima_vivari,vivari_mas_cercano,numero_vivari_cercanos,num_airbnbs_500,media_precios_airbnbs_500
0,1,2382024,3,1500,flat,rent,80.0,1.0,3,2,...,beach,0,21792.634319,Santagloria,0,15404.139470,Vivari,0,0,0.000000
1,2,2382024,1,1496,flat,rent,55.0,NaN,2,2,...,beach,0,1738.390045,Santagloria,0,643.878222,Vivari,0,22,66.826140
2,3,2382024,2,1240,flat,rent,66.0,1.0,1,1,...,beach,0,230.295035,Santagloria,2,1455.624298,Vivari,0,41,104.375729
3,4,2382024,1,750,studio,rent,30.0,0.0,0,1,...,beach,0,1871.152485,Santagloria,0,484.047977,Vivari,1,15,71.550429
4,5,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,beach,0,1306.797612,Santagloria,0,220.084909,Vivari,2,1047,109.826957


In [ ]:
df_viviendas.head()

,index,Fecha extraccion,floor,price,propertyType,operation,size,exterior,rooms,bathrooms,...,playas_mas_cercano,numero_playas_cercanos,distancia_minima_santagloria,santagloria_mas_cercano,numero_santagloria_cercanos,distancia_minima_vivari,vivari_mas_cercano,numero_vivari_cercanos,num_airbnbs_500,media_precios_airbnbs_500
0,1,2382024,3,1500,flat,rent,80.0,1.0,3,2,...,beach,0,21792.634319,Santagloria,0,15404.139470,Vivari,0,0,0.000000
1,2,2382024,1,1496,flat,rent,55.0,NaN,2,2,...,beach,0,1738.390045,Santagloria,0,643.878222,Vivari,0,22,66.826140
2,3,2382024,2,1240,flat,rent,66.0,1.0,1,1,...,beach,0,230.295035,Santagloria,2,1455.624298,Vivari,0,41,104.375729
3,4,2382024,1,750,studio,rent,30.0,0.0,0,1,...,beach,0,1871.152485,Santagloria,0,484.047977,Vivari,1,15,71.550429
4,5,2382024,NaN,1500,studio,rent,35.0,NaN,0,1,...,beach,0,1306.797612,Santagloria,0,220.084909,Vivari,2,1047,109.826957
